In [ ]:
!pip install -q yfinance pyarrow lxml

In [15]:
import pandas as pd
import requests
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Pretend the request is coming from a normal browser
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

# Extract the S&P 500 table from the HTML
sp500 = pd.read_html(StringIO(response.text))[0]

# Keep the useful columns
sp500 = sp500[
    [
        "Symbol",
        "Security",
        "GICS Sector",
        "GICS Sub-Industry"
    ]
].copy()

# Yahoo Finance formatting:
# BRK.B -> BRK-B
# BF.B  -> BF-B
sp500["Ticker"] = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
)

tickers = sp500["Ticker"].tolist()

print("Number of securities:", len(tickers))

display(sp500.head())

Number of securities: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Ticker
0,MMM,3M,Industrials,Industrial Conglomerates,MMM
1,AOS,A. O. Smith,Industrials,Building Products,AOS
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,ABT
3,ABBV,AbbVie,Health Care,Biotechnology,ABBV
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,ACN


In [16]:
data = yf.download(
    tickers=tickers,
    start="2016-01-01",
    end="2026-01-01",   # exclusive, therefore includes all of 2025
    interval="1d",
    auto_adjust=False,
    group_by="ticker",
    threads=True,
    progress=True
)

[*********************100%***********************]  502 of 503 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['HONA', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2016-01-01 -> 2026-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1451624400, endDate = 1767243600")')
ERROR:yfinance:['SNDK']: Timeout('Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
[*********************100%***********************]  502 of 503 completed

In [17]:
valid_tickers = []
invalid_tickers = []

available_tickers = set(
    data.columns.get_level_values(0)
)

for ticker in tickers:

    if ticker not in available_tickers:
        invalid_tickers.append(ticker)
        continue

    try:
        prices = data[(ticker, "Adj Close")].dropna()

        if len(prices) > 0:
            valid_tickers.append(ticker)
        else:
            invalid_tickers.append(ticker)

    except KeyError:
        invalid_tickers.append(ticker)


print("Valid tickers:", len(valid_tickers))
print("Invalid tickers:", invalid_tickers)

Valid tickers: 500
Invalid tickers: ['FDXF', 'HONA', 'SNDK']


In [18]:
data = data.loc[
    :,
    data.columns.get_level_values(0).isin(valid_tickers)
]

In [19]:
panel = (
    data
    .stack(level=0, future_stack=True)
    .rename_axis(["Date", "Ticker"])
    .reset_index()
)

panel.columns.name = None

panel["Date"] = pd.to_datetime(panel["Date"])

panel = (
    panel
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

display(panel.head())

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2016-01-04,A,41.060001,41.189999,40.340000,40.689999,37.411732,3287300.0
1,2016-01-05,A,40.730000,40.950001,40.340000,40.549999,37.283012,2587200.0
2,2016-01-06,A,40.240002,40.990002,40.049999,40.730000,37.448517,2103600.0
3,2016-01-07,A,40.139999,40.150002,38.810001,39.000000,35.857899,3504300.0
4,2016-01-08,A,39.220001,39.709999,38.470001,38.590000,35.480923,3736700.0


In [20]:
print("Number of rows:", len(panel))
print("Number of stocks:", panel["Ticker"].nunique())
print("Start:", panel["Date"].min())
print("End:", panel["Date"].max())

print("\nColumns:")
print(panel.columns.tolist())

Number of rows: 1257000
Number of stocks: 500
Start: 2016-01-04 00:00:00
End: 2025-12-31 00:00:00

Columns:
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


In [21]:
metadata_cols = [
    c for c in [
        "Ticker",
        "Security",
        "GICS Sector",
        "GICS Sub-Industry"
    ]
    if c in sp500.columns
]

metadata = sp500[metadata_cols].drop_duplicates("Ticker")

panel = panel.merge(
    metadata,
    on="Ticker",
    how="left"
)

In [22]:
panel = (
    panel
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

g = panel.groupby("Ticker")

In [23]:
panel["ret_1d"] = (
    g["Adj Close"]
    .pct_change(fill_method=None)
)

In [24]:
panel["mom_5d"] = (
    g["Adj Close"]
    .pct_change(5, fill_method=None)
)

In [25]:
panel["mom_20d"] = (
    g["Adj Close"]
    .pct_change(20, fill_method=None)
)

In [26]:
panel["mom_60d"] = (
    g["Adj Close"]
    .pct_change(60, fill_method=None)
)

In [27]:
panel["vol_5d"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .transform(
        lambda x: x.rolling(5).std()
    )
)

In [28]:
panel["vol_20d"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .transform(
        lambda x: x.rolling(20).std()
    )
)

In [29]:
panel["volume_ma20"] = (
    panel
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

In [30]:
panel["volume_ratio_20d"] = (
    panel["Volume"]
    / panel["volume_ma20"]
)

In [31]:
panel["range_pct"] = (
    (panel["High"] - panel["Low"])
    / panel["Close"]
)

In [32]:
panel["intraday_ret"] = (
    panel["Close"]
    / panel["Open"]
    - 1
)

In [33]:
panel["target"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .shift(-1)
)

In [34]:
features = [
    "ret_1d",
    "mom_5d",
    "mom_20d",
    "mom_60d",
    "vol_5d",
    "vol_20d",
    "volume_ratio_20d",
    "range_pct",
    "intraday_ret"
]